In [2]:
import pandas as pd
import numpy as np
import os
import csv

filepath = "data/non gene expression/13_CCLE_miRNA_20181103.gct"

# ── 0. File size ────────────────────────────────────────────────────────────
size_mb = os.path.getsize(filepath) / (1024 ** 2)
print(f"=== FILE SIZE: {size_mb:.1f} MB ===")

# ── 1. GCT format sniff ─────────────────────────────────────────────────────
# GCT format has:
#   Line 1: version string (#1.2 or #1.3)
#   Line 2: dimensions (rows cols) or (rows cols num_row_metadata num_col_metadata)
#   Line 3+: data (tab-separated)
print("\n=== RAW FIRST 5 LINES (GCT format check) ===")
with open(filepath, "r") as f:
    for i, line in enumerate(f):
        print(repr(line[:300]))
        if i == 4:
            break

# ── 2. Parse GCT header to get dimensions ──────────────────────────────────
print("\n=== GCT HEADER PARSE ===")
with open(filepath, "r") as f:
    version_line = f.readline().strip()
    dims_line    = f.readline().strip()

print(f"  Version: {version_line}")
print(f"  Dims:    {dims_line}")

dims = dims_line.split("\t")
n_rows = int(dims[0])
n_cols = int(dims[1])
print(f"  Declared rows: {n_rows:,}  cols: {n_cols:,}")

# ── 3. Load GCT — skip 2 header lines ──────────────────────────────────────
# GCT data starts at line 3, tab-separated
# First columns are typically: Name, Description, then sample columns
print("\n=== LOADING (skip 2 GCT header lines) ===")
df = pd.read_csv(filepath, sep="\t", skiprows=2, low_memory=False)
print(f"  Loaded shape: {df.shape}")
print(f"  Columns (first 8): {df.columns[:8].tolist()}")
print(f"  Columns (last 5):  {df.columns[-5:].tolist()}")
print(df.iloc[:5, :6].to_string())

# ── 4. ID format check ─────────────────────────────────────────────────────
print("\n=== ID FORMAT CHECK ===")
for col in df.columns[:4]:
    sample = df[col].dropna().astype(str).unique()[:5].tolist()
    print(f"  '{col}': {sample}")
    print(f"    ACH={any(v.startswith('ACH') for v in sample)}"
          f"  CVCL={any(v.startswith('CVCL') for v in sample)}"
          f"  miR={any('mir' in v.lower() or 'hsa' in v.lower() for v in sample)}")

# ── 5. miRNA naming convention ─────────────────────────────────────────────
print("\n=== miRNA NAMING ===")
for col in df.select_dtypes(include='object').columns:
    vals = df[col].dropna().astype(str)
    if vals.str.contains("mir|hsa|let", case=False).any():
        print(f"  miRNA column: '{col}'  ({df[col].nunique():,} unique)")
        print(f"  Examples: {vals.unique()[:8].tolist()}")
        print(f"  hsa- prefix:  {vals.str.startswith('hsa').sum():,}")
        print(f"  mir- prefix:  {vals.str.lower().str.startswith('mir').sum():,}")

# ── 6. Sample columns — cell line ID format ─────────────────────────────────
print("\n=== SAMPLE COLUMN FORMAT ===")
# In GCT, sample IDs are column headers after Name/Description cols
sample_cols = [c for c in df.columns if c not in ["Name", "Description", "id",
               "accession", "mir_id"] and not c.lower().startswith("name")
               and not c.lower().startswith("desc")]
print(f"  Sample columns detected: {len(sample_cols)}")
print(f"  First 5: {sample_cols[:5]}")
print(f"  Last 5:  {sample_cols[-5:]}")
has_ach  = sum(1 for c in sample_cols if c.startswith("ACH"))
has_ccle = sum(1 for c in sample_cols if "_" in c and c[0].isupper())
print(f"  ACH- format:   {has_ach}")
print(f"  CCLE-style:    {has_ccle}")

# ── 7. Null counts ─────────────────────────────────────────────────────────
print("\n=== NULL COUNTS ===")
null_counts = df.isnull().sum()
non_zero = null_counts[null_counts > 0]
print(f"  Total null cells: {null_counts.sum():,}")
if len(non_zero) == 0:
    print("  Zero nulls")
else:
    print(non_zero.head(20).to_string())

# ── 8. Numeric summary ─────────────────────────────────────────────────────
print("\n=== NUMERIC SUMMARY ===")
numeric_cols = df.select_dtypes(include='number').columns
print(f"  Numeric columns: {len(numeric_cols)}")
if len(numeric_cols) > 0:
    vals = df[numeric_cols].values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"  Global min:    {vals.min():.4f}")
    print(f"  Global max:    {vals.max():.4f}")
    print(f"  Global mean:   {vals.mean():.4f}")
    print(f"  Global median: {np.median(vals):.4f}")
    print(f"  Global std:    {vals.std():.4f}")
    zero_pct = (vals == 0).mean() * 100
    print(f"  Zero values:   {zero_pct:.1f}%")
    neg_pct  = (vals < 0).mean() * 100
    print(f"  Negative values: {neg_pct:.1f}%")
    if vals.min() < 0:
        print("  → Negative values present — suggests log2-transformed (log2 RPM)")
    elif vals.max() < 25:
        print("  → Suggests log2-transformed")
    else:
        print("  → Suggests raw or RPM counts")

# ── 9. Missingness pattern ─────────────────────────────────────────────────
print("\n=== MISSINGNESS PATTERN ===")
if len(numeric_cols) > 0:
    row_null = df[numeric_cols].isnull().mean(axis=1)
    col_null = df[numeric_cols].isnull().mean(axis=0)
    print(f"  Mean % missing per row (miRNA): {row_null.mean()*100:.1f}%")
    print(f"  miRNAs with >50% missing:       {(row_null > 0.5).sum():,}")
    print(f"  miRNAs with 0% missing:         {(row_null == 0).sum():,}")
    print(f"  Mean % missing per col (sample):{col_null.mean()*100:.1f}%")
    print(f"  Samples with >50% missing:      {(col_null > 0.5).sum():,}")
    print(f"  Samples with 0% missing:        {(col_null == 0).sum():,}")

# ── 10. DepMap overlap ─────────────────────────────────────────────────────
print("\n=== DEPMAP OVERLAP ===")
profiles  = pd.read_csv("data/nomenclature/8_DepMap_OmicsProfiles.csv")
rna_ach   = set(profiles[profiles["Datatype"] == "rna"]["ModelID"].dropna())
mirna_ids = set(sample_cols)

# Try direct ACH- match first
ach_match = mirna_ids & rna_ach
print(f"  Direct ACH- overlap with RNA profiles: {len(ach_match):,}")

# If no ACH- match, sample cols are probably CCLE-style — bridge via sample_info
if len(ach_match) == 0:
    print("  No direct ACH- match — trying CCLE_Name bridge via sample_info")
    sample_info = pd.read_csv("data/nomenclature/9_DepMap_sample_info.csv")
    ccle_to_ach = dict(zip(sample_info["CCLE_Name"], sample_info["DepMap_ID"]))
    mapped = {c: ccle_to_ach.get(c) for c in sample_cols if c in ccle_to_ach}
    print(f"  Sample cols matching CCLE_Name:        {len(mapped):,} / {len(sample_cols):,}")
    mapped_ach = set(v for v in mapped.values() if v)
    overlap_rna = mapped_ach & rna_ach
    print(f"  Of those, also have RNA data:          {len(overlap_rna):,}")

=== FILE SIZE: 8.7 MB ===

=== RAW FIRST 5 LINES (GCT format check) ===
'#1.2\t\n'
'734\t954\t\n'
'Name\tDescription\tDMS53_LUNG\tSW1116_LARGE_INTESTINE\tNCIH1694_LUNG\tP3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE\tHUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE\tUMUC3_URINARY_TRACT\tHOS_BONE\tHUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE\tAML193_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE\tRVH421_SKIN\tNCIH1184_LUNG\tHCC2157_BREAST\t'
'nmiR00001.1\thsa-let-7a\t  4362.58000\t  5191.50000\t 24991.05000\t  3253.83000\t   225.28000\t 35051.17000\t  5706.63000\t  1821.46000\t 31330.15000\t  6376.17000\t 10679.84000\t 15209.77000\t 49342.72000\t 16397.23000\t 25512.29000\t 65349.28000\t 35065.07000\t  9938.79000\t 26199.57000\t 25080.76000\t  3329.48000\t  80'
'nmiR00002.1\thsa-let-7b\t   187.44000\t   868.22000\t  5066.09000\t    74.21000\t    35.74000\t  1014.65000\t  1211.27000\t    22.54000\t  2082.84000\t  1228.14000\t  2924.39000\t  2303.30000\t  1628.01000\t   221.77000\t  1672.54000\t  7627.99000\t  3256.75

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/non gene expression/13_CCLE_miRNA_20181103.gct",
                 sep="\t", skiprows=2, low_memory=False)

sample_info = pd.read_csv("data/nomenclature/9_DepMap_sample_info.csv")
sample_cols = [c for c in df.columns if c not in ["Name", "Description"]]

# ── 1. Identify the 4 unmatched CCLE columns and the 1 ACH- column ─────────
print("=== UNMATCHED SAMPLE COLUMNS ===")
ccle_names = set(sample_info["CCLE_Name"].dropna())
unmatched = [c for c in sample_cols if c not in ccle_names and not c.startswith("ACH")]
ach_cols   = [c for c in sample_cols if c.startswith("ACH")]
print(f"  Unmatched (not CCLE_Name, not ACH-): {unmatched}")
print(f"  ACH- columns: {ach_cols}")

# ── 2. Check the 80 non-hsa miRNAs ─────────────────────────────────────────
print("\n=== NON-HSA miRNA CHECK ===")
non_hsa = df[~df["Description"].str.startswith("hsa", na=False)]
print(f"  Total non-hsa rows: {len(non_hsa)}")
print(f"  Description examples: {non_hsa['Description'].unique()[:15].tolist()}")
print(f"  Name examples:        {non_hsa['Name'].unique()[:15].tolist()}")

# Are they high or low expression — worth keeping?
numeric_cols = df.select_dtypes(include='number').columns
non_hsa_vals = df.loc[~df["Description"].str.startswith("hsa", na=False), numeric_cols]
hsa_vals     = df.loc[df["Description"].str.startswith("hsa", na=False), numeric_cols]
print(f"\n  Non-hsa mean expression: {non_hsa_vals.values.mean():.2f}")
print(f"  hsa mean expression:     {hsa_vals.values.mean():.2f}")
print(f"  Non-hsa median:          {np.median(non_hsa_vals.values):.2f}")
print(f"  hsa median:              {np.median(hsa_vals.values):.2f}")

# ── 3. Variance — top discriminating miRNAs ─────────────────────────────────
print("\n=== TOP 10 MOST VARIABLE miRNAs ===")
log_df = np.log2(df[numeric_cols] + 1)
mirna_std = log_df.std(axis=1)
mirna_std.index = df["Description"]
print(mirna_std.sort_values(ascending=False).head(10).to_string())

print("\n=== TOP 10 LEAST VARIABLE miRNAs ===")
print(mirna_std.sort_values(ascending=False).tail(10).to_string())

=== UNMATCHED SAMPLE COLUMNS ===
  Unmatched (not CCLE_Name, not ACH-): ['KE97_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE', 'NCIH1339_LUNG', 'NCIH684_LIVER', 'COLO699_LUNG']
  ACH- columns: ['ACHN_KIDNEY']

=== NON-HSA miRNA CHECK ===
  Total non-hsa rows: 80
  Description examples: ['bkv-miR-B1-3p+jcv-miR-J1-3p', 'bkv-miR-B1-5p', 'ebv-miR-BART1-3p', 'ebv-miR-BART1-5p', 'ebv-miR-BART10', 'ebv-miR-BART11-3p', 'ebv-miR-BART11-5p', 'ebv-miR-BART12', 'ebv-miR-BART13', 'ebv-miR-BART14', 'ebv-miR-BART15', 'ebv-miR-BART16', 'ebv-miR-BART17-3p', 'ebv-miR-BART17-5p', 'ebv-miR-BART18-3p']
  Name examples:        ['nmiR00717.1', 'nmiR00718.1', 'nmiR00719.1', 'nmiR00720.1', 'nmiR00721.1', 'nmiR00722.1', 'nmiR00723.1', 'nmiR00724.1', 'nmiR00725.1', 'nmiR00726.1', 'nmiR00727.1', 'nmiR00728.1', 'nmiR00729.1', 'nmiR00730.1', 'nmiR00731.1']

  Non-hsa mean expression: 37.75
  hsa mean expression:     421.25
  Non-hsa median:          15.57
  hsa median:              23.05

=== TOP 10 MOST VARIABLE miRNAs ===
D